## DATA INGESTION PIPELINE - TTC Delay Prediction Project

#### PURPOSE:
##### This script downloads raw datasets required for the ML project:
+ 1. TTC Bus Delay data (historical incidents)
+ 2. Toronto Weather data (Environment Canada)
+ 3. TTC GTFS (routes + schedules)

#### OUTPUT:
##### All datasets are saved as CSV/XLSX files inside the /data folder, organized into subfolders per dataset.
###### These files are later used for feature engineering and ML training.

In [105]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [106]:
import os
import shutil
import requests
import zipfile
import time
import pandas as pd
from pathlib import Path
from io import StringIO
import sys

#### CONFIG

In [107]:
# Base API for Toronto Open Data (CKAN)
BASE_URL = "https://ckan0.cf.opendata.inter.prod-toronto.ca"
PACKAGE_SHOW_URL = f"{BASE_URL}/api/3/action/package_show"

# Folder structure for raw datasets
DATA_DIR = Path("data/raw")
TTC_DIR = DATA_DIR/"ttc_bus"
WEATHER_DIR = DATA_DIR/"weather"
GTFS_DIR = DATA_DIR/"ttc_gtfs"

# TTC bus delay files for following years
TARGET_YEARS = range(2022, 2026)

# Weather station + date range (Toronto station)
WEATHER_STATION_ID = 48549
WEATHER_START_YEAR = 2022
WEATHER_END_YEAR = 2025

#### HELPER FUNCTIONS

In [108]:
# Create a folder (and parents) if it doesn't already exist
def make_folder(path):
    os.makedirs(path, exist_ok=True)

# Get dataset file list from Toronto Open Data (CKAN API)
def get_ckan_resources(package_id):
    response = requests.get(PACKAGE_SHOW_URL, params={"id": package_id}, timeout=30)
    response.raise_for_status()
    return response.json()["result"]["resources"]

# Download a file from a URL and save it locally
# Uses streaming to handle large GTFS files safely
def download_file(url, path, overwrite=False):
    if path.exists() and not overwrite:
        print(f"Skipping: {path.name}")
        return

    print(f"Downloading: {path.name}")
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()

        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)

#### TTC BUS DELAYS (HISTORICAL)

In [109]:
# Downloads TTC bus delay datasets
# Contains historical delay records (2022–2025)
# Used ONLY for training the ML model
# NOT updated weekly because it is fixed historical data
def download_ttc_bus_delays():
    make_folder(TTC_DIR)
    resources = get_ckan_resources("ttc-bus-delay-data")

    for resource in resources:
        name = resource.get("name", "")
        url = resource.get("url", "")

        # keep only required years
        if not any(str(year) in name or str(year) in url for year in TARGET_YEARS):
            continue

        # only data files
        if not url.lower().endswith((".csv", ".xlsx")):
            continue

        filename = url.split("/")[-1]
        download_file(url, TTC_DIR / filename)

    print("TTC bus delay download complete.")

#### TTC GTFS (ROUTES & SCHEDULES) (WEEKLY UPDATE)

In [110]:
# Downloads TTC GTFS schedule data
# GTFS is static infrastructure data (routes, stops, shapes).
# It is refreshed periodically (monthly or per project cycle)
# to ensure alignment with TTC system updates, but does not affect training data.
def download_ttc_gtfs():
    make_folder(GTFS_DIR)
    resources = get_ckan_resources("ttc-routes-and-schedules")

    for resource in resources:
        url = resource.get("url", "")

        if not url:
            continue

        filename = url.split("/")[-1]
        download_file(url, GTFS_DIR / filename, overwrite=True)

    # Extract GTFS zip files
    for zip_path in GTFS_DIR.glob("*.zip"):
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(GTFS_DIR)
        zip_path.unlink()

    print("GTFS download complete.")

#### WEATHER DATA  (HISTORICAL)

In [112]:
# Downloads full weather dataset (2022–2025) and merges into one file
# Historical weather data used for training correlations between weather conditions and bus delays
# Not refreshed in weekly pipeline
def fetch_weather_month(station_id, year, month):
    # builds the URL for one month of data from Environment Canada
    url = ("https://climate.weather.gc.ca/climate_data/bulk_data_e.html"
           f"?format=csv&stationID={station_id}&Year={year}&Month={month}"
           f"&Day=1&timeframe=1")

    try:
        text = requests.get(url, timeout=60).text
    except requests.RequestException as e:
        print(f"  Failed to fetch {year}-{month:02d}: {e}")
        return None

    # skip metadata header
    header_row = next((i for i, line in enumerate(text.split("\n")) if "Climate ID" in line), None)
    if header_row is None:
        return None

    df = pd.read_csv(StringIO(text), skiprows=header_row)
    df["year"] = year
    df["month"] = month
    return df


def download_weather_data():
    make_folder(WEATHER_DIR)
    output_file = WEATHER_DIR / "toronto_weather_2022_2025.csv"

    # Skip if we already exists
    if output_file.exists():
        print("Skipping (already exists): toronto_weather_2022_2025.csv")
        return

    # pull one month at a time and collect into a list of DataFrames
    all_data = []
    for year in range(WEATHER_START_YEAR, WEATHER_END_YEAR + 1):
        for month in range(1, 13):
            df = fetch_weather_month(WEATHER_STATION_ID, year, month)
            if df is not None:
                all_data.append(df)

    if not all_data:
        print("No weather data retrieved; nothing to save.")
        return

    # combine all months into one DataFrame and save as a single CSV
    weather_df = pd.concat(all_data, ignore_index=True)
    if "Date/Time" in weather_df.columns:
        weather_df["Date/Time"] = pd.to_datetime(weather_df["Date/Time"], errors="coerce")

    weather_df.to_csv(output_file, index=False)
    print("Weather data download complete.")

#### MAIN PIPELINE

In [116]:
def run_full_ingestion():
    make_folder(DATA_DIR)
    download_ttc_bus_delays()
    download_ttc_gtfs()
    download_weather_data()
    print("Full ingestion complete")

if __name__ == "__main__":
    run_full_ingestion()

Skipping: ttc-bus-delay-data-2022.xlsx
Skipping: ttc-bus-delay-data-2023.xlsx
Skipping: ttc-bus-delay-data-2024.xlsx
Skipping: ttc-bus-delay-data-since-2025.csv
TTC bus delay download complete.
Downloading: opendata_ttc_schedules.zip
GTFS download complete.
Skipping (already exists): toronto_weather_2022_2025.csv
Full ingestion complete
